# 030 — Flux cycle calculation

This tutorial computes CO2 + H2O fluxes from quality-controlled chamber
concentration cycles. It runs end-to-end on the **bundled synthetic
sample** (no setup required) — set ``PALMWTC_DATA_DIR`` to point at your
own QC parquet to use real data instead.

What you'll see:
1. Resolve I/O paths (config layered: kwargs → env → yaml → bundled).
2. Run the ``"flux"`` pipeline step (under the hood: cycle identification,
   linear-fit slope per cycle, scoring, optional ML outlier flagging).
3. Plot the resulting per-cycle flux time series and a diurnal heatmap.
4. Inspect the cycles dataframe for downstream calibration.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

from palmwtc.config import DataPaths
from palmwtc.pipeline import run_step
from palmwtc.viz import set_style

set_style()
pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 20)


## 1. Resolve I/O paths

`DataPaths.resolve()` walks: explicit kwargs → `PALMWTC_DATA_DIR` env →
`palmwtc.yaml` → bundled synthetic sample. The last layer always succeeds,
so this notebook runs even on a fresh `pip install palmwtc` with no setup.


In [ ]:
paths = DataPaths.resolve()
print(paths.describe())


## 2. Run the flux step

`run_step("flux")` does the work: load QC parquet → discover chambers from
`CO2_C<n>` columns → for each chamber, prepare data + identify cycles +
fit slopes + score quality → write `01_chamber_cycles.csv`.

This is one library call, fully testable, no notebook-cell-resident logic.


In [ ]:
result = run_step("flux", paths)
print(f"Step status: {'OK' if result.ok else 'FAILED'}")
print(f"Elapsed:      {result.elapsed_seconds:.1f}s")
print(f"Rows in:      {result.rows_in:,}")
print(f"Rows out:     {result.rows_out}")
print(f"Artefact:     {result.artefacts[0]}")
print(f"Chambers:     {result.metrics.get('chambers')}")


## 3. Inspect the cycle output


In [ ]:
cycles = pd.read_csv(result.artefacts[0])
print(f"{len(cycles)} cycles across {cycles['chamber'].nunique()} chamber(s)")
cycles[["chamber", "cycle_id", "flux_date", "flux_slope", "r2", "qc_flag", "flux_absolute"]].head()


## 4. Plot the per-cycle flux series

The synthetic sample only produces a handful of cycles (it's 1 week of
toy data). Real LIBZ data yields thousands — try setting
`PALMWTC_DATA_DIR` to a real chamber dataset.


In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
for chamber, group in cycles.groupby("chamber"):
    ax.scatter(
        pd.to_datetime(group["flux_date"]),
        group["flux_absolute"],
        label=f"Chamber {chamber}",
        s=60,
        alpha=0.7,
    )
ax.axhline(0, color="grey", linewidth=0.6, linestyle="--")
ax.set_xlabel("Date")
ax.set_ylabel("Absolute CO2 flux  (μmol m⁻² s⁻¹)")
ax.set_title("Per-cycle CO2 flux (synthetic sample)")
ax.legend()
plt.tight_layout()
plt.show()


## 5. Cycle-quality summary

`qc_flag` is the per-cycle pass/fail flag (0 = pass, 1 = warn, 2 = fail).
`r2` is the linear-fit goodness on the closed-phase concentration ramp;
high R² + low NRMSE + appropriate SNR → cycle accepted into calibration windows.


In [ ]:
cycles[["chamber", "qc_flag", "r2", "nrmse", "snr"]].groupby("chamber").describe()


## Next

- **031 / 032** — promote high-confidence cycles into calibration windows
  (`run_step("windows", paths)`).
- **033** — validate against literature ecophysiology bounds
  (`run_step("validation", paths)`).
- **CLI shortcut** — `palmwtc run` runs all four steps end-to-end.
